# Imports

In [ ]:
import torch
import sys
import os
from torch import Tensor
sys.path.append('..')
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go
from definitions import EXPERIMENTS_DIR, ROOT_DIR
from src.commands.train import Trainer, Config
from torch_geometric.data import HeteroData
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    hamming_loss,
)
from PIL import ImageColor

In [2]:
px.defaults.template = 'seaborn'

In [3]:
dir = os.path.join(ROOT_DIR, 'assets', 'articles_classification_results')

# Load the results

In [4]:
# TASK = 'hierarchical_classification'
EXPERIMENT = 'nantes-uni-2020_e5-v2'
CORPUS = 'nantes-uni-2020'
EXPERIMENT_DIR = os.path.join(EXPERIMENTS_DIR, EXPERIMENT)
RESULTS_DIR = os.path.join(EXPERIMENT_DIR, 'results')
HISTORIES_DIR = os.path.join(EXPERIMENT_DIR, 'histories')
CONFIG_PATH = os.path.join(EXPERIMENT_DIR, 'config.json')
SPLIT_PATH = os.path.join(EXPERIMENT_DIR, 'split_sub_dsis_4.pt')

In [5]:
history = pd.concat([
    pd.read_csv(os.path.join(HISTORIES_DIR, filename))
    for filename in os.listdir(HISTORIES_DIR)
])
history = history \
    .groupby(['epoch','phase','config','trial']) \
    .mean() \
    .reset_index()
history.head()

,epoch,phase,config,trial,subdomain_guidance_loss,lab_guidance_loss,loss,subdomain_guidance_acc,subdomain_guidance_f1,lab_guidance_acc,lab_guidance_f1
0,0,train,Conv_Attn_null_False,0,0.182479,0.109717,0.292196,0.930327,0.300421,0.959082,0.386406
1,0,train,Conv_Attn_null_False,1,0.180180,0.109062,0.289242,0.929941,0.302938,0.960134,0.385479
2,0,train,Conv_Attn_null_False,2,0.183625,0.110667,0.294292,0.929027,0.305162,0.958357,0.380009
3,0,train,Conv_Attn_null_False,3,0.179828,0.106661,0.286488,0.931642,0.311706,0.959970,0.390599
4,0,train,Conv_Max_null_False,0,0.191868,0.122028,0.313896,0.924226,0.275262,0.955364,0.360757


In [6]:
history_mean = history \
    .drop(columns = 'trial') \
    .groupby(['epoch','phase','config']).mean() \
    .add_suffix('_mean')

history_std = history \
    .drop(columns = 'trial') \
    .groupby(['epoch','phase','config']).std() \
    .add_suffix('_std')

In [7]:
history = history_mean.join(other = history_std).reset_index()
history.head()

,epoch,phase,config,subdomain_guidance_loss_mean,lab_guidance_loss_mean,loss_mean,subdomain_guidance_acc_mean,subdomain_guidance_f1_mean,lab_guidance_acc_mean,lab_guidance_f1_mean,subdomain_guidance_loss_std,lab_guidance_loss_std,loss_std,subdomain_guidance_acc_std,subdomain_guidance_f1_std,lab_guidance_acc_std,lab_guidance_f1_std
0,0,train,Conv_Attn_null_False,0.181528,0.109027,0.290555,0.930234,0.305057,0.959386,0.385623,0.001827,0.001709,0.003412,0.001085,0.004838,0.000827,0.004355
1,0,train,Conv_Max_null_False,0.191591,0.118736,0.310327,0.925404,0.275156,0.955937,0.358965,0.001875,0.002799,0.004072,0.001076,0.005372,0.000726,0.007532
2,0,train,Conv_Mean_null_False,0.181363,0.110226,0.291589,0.929155,0.301665,0.959169,0.382360,0.001996,0.003912,0.005620,0.000555,0.003728,0.001735,0.010002
3,0,val,Conv_Attn_null_False,0.191980,0.090377,0.282357,0.935573,0.347628,0.975142,0.448788,0.015969,0.016021,0.030808,0.002812,0.010874,0.002927,0.014464
4,0,val,Conv_Max_null_False,0.186668,0.092413,0.279081,0.934949,0.325914,0.971418,0.433204,0.019009,0.008110,0.024565,0.003382,0.032953,0.002248,0.017099


# Load Data

In [8]:
config = Config(experiment=EXPERIMENT)
trainer = Trainer(experiment=config.experiment, config=config.training_config)
configs = trainer.get_configs()

In [9]:
data = trainer.load_data()
# data = trainer.filter_data(data)
# data = trainer.keep_largest_cc(data) if trainer.config.keep_largest_cc else data
data.split = trainer.create_or_load_split(data)
data['Article'].split = data.split

In [10]:
metadata = data.raw_metadata

# Learning Curves

In [11]:
def line_plot(
    df : pd.DataFrame,
    metric : str
) -> None:
    
    fig = go.Figure()

    colors = px.colors.qualitative.Plotly
    exprs = df['config'].unique()

    for i, expr in enumerate(exprs):

        history_val = df.query(f"phase == 'val' and config == '{expr}'")
        color = colors[i % len(colors)]

        fig.add_trace(
            go.Scatter(
                name = expr,
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean'],
                mode='lines',
                line=dict(color=color),
            )
        )

        fig.add_trace(
            go.Scatter(
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean']+history_val[f'{metric}_std'],
                mode='lines',
                marker=dict(color="#444"),
                line=dict(width=0),
                showlegend=False
            )
        )

        fig.add_trace(
            go.Scatter(
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean']-history_val[f'{metric}_std'],
                marker=dict(color="#444"),
                line=dict(width=0),
                mode='lines',
                fillcolor=f'rgb{ImageColor.getcolor(color, "RGB")}'.replace('rgb','rgba').replace(')',',0.2)'),
                fill='tonexty',
                showlegend=False
            )
        )

    fig.update_layout(
        xaxis_title='Epoch',
        yaxis_title=metric.capitalize(),
        width = 960,
        height = 800
    )

    fig.show()

In [12]:
line_plot(history, 'loss')

In [13]:
line_plot(history, 'subdomain_guidance_acc')

In [14]:
line_plot(history, 'subdomain_guidance_f1')

In [15]:
line_plot(history, 'lab_guidance_acc')

In [16]:
line_plot(history, 'lab_guidance_f1')

# Calculate other metrics

## Load Predictions

In [17]:
def load_results(results_dir : str) -> dict[tuple[str, str], Tensor]:

    results = dict()

    for filename in os.listdir(results_dir):

        parts = filename.split('.')[0].split('_')
        trial = parts[-1]
        expr = '_'.join(parts[1:-1])

        results[(expr,trial)] = torch.load(os.path.join(RESULTS_DIR, filename), map_location='cpu', weights_only=False)

    return results

In [18]:
results = load_results(RESULTS_DIR)

## Label Wise & Row Wise Metrics

In [19]:
def display_mean_std_df(df : pd.DataFrame) -> pd.DataFrame:

    columns = set([
        col.replace('_mean','')
        for col in df.columns if 'mean' in col
    ])

    new_df = pd.DataFrame()

    for col in df.columns:

        if not col.endswith('_mean') and not col.endswith('_std'):
            new_df[col] = df[col]

    for col in columns:
        new_df[col] = df[f'{col}_mean'].apply(lambda x : f'{x:.2%}') + ' ± ' + df[f'{col}_std'].apply(lambda x : f'{x:.2%}')

    return new_df

In [20]:
def get_experiment_params(expr : str) -> dict:

    parts = expr.split('_')
        
    return {
        "GNN Variant" : parts[0],
        "Readout Function" : parts[1],
        "Pooling Layer" : parts[2],
        "Undersampling" : parts[3]
    }

In [21]:
def get_label_wise_metrics(
    label_name : str,
    data : HeteroData,
    results : dict[tuple[str,str],Tensor],
    labels : list[str]
) -> tuple[pd.DataFrame,pd.DataFrame]:

    label_wise_metrics = []

    n_labels = data[f'y_{label_name}'] if label_name != 'lab' else data[f'y_{label_name}'][:,:-1]
    n_labels = n_labels.size(-1)

    for label in range(n_labels):
            
        for (expr,trial),result in results.items():

            y = data[f'y_{label_name}'][data.split == int(trial[-1])]
            y_hat = result[label_name][data.split == int(trial[-1])].sigmoid()
            y_hard = y_hat.gt(0.5).long()
            
            label_wise_metrics.append({
                "team" : label,
                "expr" : expr,
                "trial" : trial,
                "accuracy" : accuracy_score(y[:,label], y_hard[:,label]),
                "f1_score" : f1_score(y[:,label], y_hard[:,label]),
                "precision" : precision_score(y[:,label], y_hard[:,label]),
                "recall" : recall_score(y[:,label], y_hard[:,label]),
                "auc" : roc_auc_score(y[:,label], y_hat[:,label])
            })

    label_wise_metrics = pd.DataFrame(label_wise_metrics)
    label_wise_metrics['team'] = label_wise_metrics['team'].apply(labels.__getitem__)
    label_wise_metrics['expr'] = label_wise_metrics['expr'].apply(str)

    label_wise_metrics_agg = label_wise_metrics.drop(columns="team").groupby(["expr","trial"]).mean().reset_index().sort_values(by = 'expr')

    label_wise_metrics_mean = label_wise_metrics_agg \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .mean().add_suffix('_mean')

    label_wise_metrics_std = label_wise_metrics_agg \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .std().add_suffix('_std')

    label_wise_metrics_agg = label_wise_metrics_mean.join(label_wise_metrics_std).reset_index()

    return label_wise_metrics_agg,label_wise_metrics

In [22]:
def get_row_wise_metrics(
    label : str,
    data : HeteroData,
    results : dict[tuple[str,str],Tensor],
) -> tuple[pd.DataFrame,pd.DataFrame]:

    row_wise_metrics = []

    for (expr,trial),result in results.items():

        y = data[f'y_{label}'][data.split == int(trial[-1])]
        y_hat = result[label][data.split == int(trial[-1])].sigmoid()
        y_hard = y_hat.gt(0.5).long()

        if label == 'lab':
            y = y[:,:-1]
            y_hat = y_hat[:,:-1]
            y_hard = y_hard[:,:-1]

        row_wise_metrics.append({
            "expr" : expr,
            "trial" : trial,
            "accuracy" : accuracy_score(y, y_hard),
            "f1_score" : f1_score(y, y_hard , average='macro'),
            "precision" : precision_score(y, y_hard , average='macro'),
            "recall" : recall_score(y, y_hard , average='macro'),
            "auc" : roc_auc_score(y, y_hat , average='macro'),
            "hamming_loss" : hamming_loss(y, y_hard)
        })

    row_wise_metrics = pd.DataFrame(row_wise_metrics)

    row_wise_metrics_mean = row_wise_metrics \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .mean().add_suffix('_mean')

    row_wise_metrics_std = row_wise_metrics \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .std().add_suffix('_std')

    row_wise_metrics_agg = row_wise_metrics_mean.join(row_wise_metrics_std).reset_index()

    return row_wise_metrics_agg,row_wise_metrics

In [23]:
label_wise_metrics_agg,label_wise_metrics = get_label_wise_metrics('subdomain', data['Article'], results, metadata['labels']['subdomain'])
display_mean_std_df(label_wise_metrics_agg)

,expr,auc,recall,accuracy,f1_score,precision
0,Conv_Attn_null_False,97.09% ± 0.22%,69.87% ± 1.83%,95.78% ± 0.25%,73.94% ± 1.88%,79.25% ± 2.26%
1,Conv_Max_null_False,96.79% ± 0.33%,68.64% ± 2.63%,95.66% ± 0.21%,72.90% ± 1.91%,78.73% ± 1.37%
2,Conv_Mean_null_False,97.09% ± 0.13%,66.14% ± 2.56%,95.79% ± 0.24%,72.55% ± 2.12%,81.96% ± 1.96%


In [24]:
row_wise_metrics_agg,row_wise_metrics = get_row_wise_metrics('subdomain', data['Article'], results)
metrics = display_mean_std_df(row_wise_metrics_agg)
metrics

,expr,auc,recall,accuracy,f1_score,hamming_loss,precision
0,Conv_Attn_null_False,97.09% ± 0.22%,69.87% ± 1.83%,52.62% ± 2.15%,73.94% ± 1.88%,4.22% ± 0.25%,79.25% ± 2.26%
1,Conv_Max_null_False,96.79% ± 0.33%,68.64% ± 2.63%,52.10% ± 1.51%,72.90% ± 1.91%,4.34% ± 0.21%,78.73% ± 1.37%
2,Conv_Mean_null_False,97.09% ± 0.13%,66.14% ± 2.56%,52.91% ± 1.67%,72.55% ± 2.12%,4.21% ± 0.24%,81.96% ± 1.96%


In [25]:
label_wise_metrics_agg,label_wise_metrics = get_label_wise_metrics('lab', data['Article'], results, metadata['labels']['lab'])
display_mean_std_df(label_wise_metrics_agg)

,expr,auc,recall,accuracy,f1_score,precision
0,Conv_Attn_null_False,98.90% ± 0.37%,83.04% ± 4.34%,98.10% ± 0.37%,86.24% ± 3.04%,90.87% ± 2.55%
1,Conv_Max_null_False,98.68% ± 0.32%,82.29% ± 2.88%,97.90% ± 0.27%,85.26% ± 2.15%,89.46% ± 1.89%
2,Conv_Mean_null_False,98.89% ± 0.30%,80.13% ± 4.11%,97.99% ± 0.32%,84.61% ± 3.62%,91.27% ± 3.14%


In [26]:
row_wise_metrics_agg,row_wise_metrics = get_row_wise_metrics('lab', data['Article'], results)
metrics = display_mean_std_df(row_wise_metrics_agg)
metrics

,expr,auc,recall,accuracy,f1_score,hamming_loss,precision
0,Conv_Attn_null_False,98.90% ± 0.37%,83.04% ± 4.34%,87.18% ± 2.08%,86.24% ± 3.04%,1.90% ± 0.37%,90.87% ± 2.55%
1,Conv_Max_null_False,98.68% ± 0.32%,82.29% ± 2.88%,85.68% ± 2.00%,85.26% ± 2.15%,2.10% ± 0.27%,89.46% ± 1.89%
2,Conv_Mean_null_False,98.89% ± 0.30%,80.13% ± 4.11%,85.76% ± 2.49%,84.61% ± 3.62%,2.01% ± 0.32%,91.27% ± 3.14%


In [27]:
hparams = metrics.apply(lambda x : pd.Series(get_experiment_params(x['expr'])), axis = 1)
metrics = hparams.join(metrics, lsuffix="hparams")
metrics['Experiment'] = metrics['expr']
metrics = metrics.drop(columns = ['expr'])
metrics = metrics[[metrics.columns[-1], *metrics.columns[:-1]]]
# metrics.to_csv(os.path.join(dir, "hierarchical_teams_metrics.xlsx"), index = False)
metrics

,Experiment,GNN Variant,Readout Function,Pooling Layer,Undersampling,auc,recall,accuracy,f1_score,hamming_loss,precision
0,Conv_Attn_null_False,Conv,Attn,null,False,98.90% ± 0.37%,83.04% ± 4.34%,87.18% ± 2.08%,86.24% ± 3.04%,1.90% ± 0.37%,90.87% ± 2.55%
1,Conv_Max_null_False,Conv,Max,null,False,98.68% ± 0.32%,82.29% ± 2.88%,85.68% ± 2.00%,85.26% ± 2.15%,2.10% ± 0.27%,89.46% ± 1.89%
2,Conv_Mean_null_False,Conv,Mean,null,False,98.89% ± 0.30%,80.13% ± 4.11%,85.76% ± 2.49%,84.61% ± 3.62%,2.01% ± 0.32%,91.27% ± 3.14%


- Visualizations

In [28]:
def bar_plot(df : pd.DataFrame, metric : str) -> None:

    fig = px.bar(
        data_frame = df.sort_values(by = f'{metric}_mean'),
        x = 'expr',
        y = f'{metric}_mean',
        barmode = 'group',
        error_y = f'{metric}_std',
        text = f'{metric}_mean',
        width = 950,
        height = 800
    )

    fig.update_layout(
        xaxis_title = 'Experiment',
        yaxis_title = metric.capitalize().replace('_',' '),
    )

    fig.update_traces(
        textposition = 'outside',
        texttemplate = '%{text:.2%}'
    )

    fig.show()

In [29]:
bar_plot(label_wise_metrics_agg, 'accuracy')

In [30]:
bar_plot(label_wise_metrics_agg, 'f1_score')

In [31]:
bar_plot(row_wise_metrics_agg, 'accuracy')